# 03.3 HDBSCAN + GLOSH

Вход (артефакты из 03.1):
- `split_metadata.json`, `clip_bounds.json`, `scaler.joblib`, `contract.json`.

Выход:
- 36 артефактов: `hdb_pca_<group>_<seed>.joblib` (4 группы x 3 seeds), `hdb_scaler_<group>_<seed>.joblib` (x12), `hdb_<group>_<seed>.joblib` (x12).
- `hdb_window_scores.parquet`: window-level scores.
- `hdb_scores.parquet`: point-level scores (с row_id для merge в 03.5).
- `hdb_stability.json`: Jaccard / Pearson stability метрики.

## Замечания по реализации

`approximate_predict_scores` возвращает один массив (была ошибка распаковки). Добавлен `np.clip(0, 1)` для numerical safety.

Pass A обрабатывает только полные рейсы: последний flight_id очередного batch'а переносится на следующий через `carry` DataFrame. Это нужно, чтобы ts_sec и индексы окон считались от истинного начала рейса, а не от первой точки batch.

`finite_share` считается через подсчёт finite-точек в окне. Флаг `window_finite=1` означает, что ВСЕ точки окна имеют finite model features (а не что pandas mean/std не вернул NaN из-за skipna).

`DQ_HARD_SHARE_TRAINING_MAX = 0.0`: в training pool не попадает ни одна dq_hard точка.

Маппинг window-to-point идёт через явный ключ `(flight_id, window_idx, window_offset)`, без post-filter, чтобы исключить cartesian explosion.

Top-k preview работает через streaming nlargest, чтобы не читать весь parquet в RAM.

В Jaccard top-k используется `np.argpartition` вместо полной сортировки.

Sanity-проверка: ключи `(flight_id, window_idx, window_offset)` должны быть уникальны (иначе carryover сломался).

## Четыре phase groups

- climb (phase 2)
- cruise (phase 3)
- descent (phase 4)
- terminal (phases 0, 1, 5, 6: ground + takeoff + approach + landing)

Семь отдельных HDBSCAN дали бы over-fitting к мелким классам. Одна глобальная HDBSCAN кластеризует фазы, а не аномалии (известно из v2 диагностики). Четыре группы это компромисс.

## Логика трёх проходов

Pass A (построение окон):
- carry full flights между batches;
- для каждого complete-flight chunk: clip+scale, затем ts_sec, затем window_a/window_b, затем aggregate;
- сохраняем окна и point chunks (row_id consistent с if_scores).

Train HDBSCAN per group, 3 stability samples (random subsets of clean train_core).

Score windows:
- main: все eligible окна;
- s2/s3: только calibration_val для stability.

Window-level percentile против clean calibration_val.

Маппинг окон на точки через offset key.

Stability analysis на уровне окон.
Top preview через streaming top-k.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time
import glob
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import hdbscan
import joblib

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')

INPUT_PATH = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')

# 03.1 artifacts
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')
CLIP_BOUNDS_PATH = os.path.join(ARTIFACTS_DIR, 'clip_bounds.json')
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')
CONTRACT_PATH = os.path.join(ARTIFACTS_DIR, 'contract.json')

# 03.3 outputs
HDB_WINDOW_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'hdb_window_scores.parquet')
HDB_POINT_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'hdb_scores.parquet')
HDB_STABILITY_PATH = os.path.join(ARTIFACTS_DIR, 'hdb_stability.json')
HDB_WINDOWS_TEMP_DIR = os.path.join(ARTIFACTS_DIR, 'hdb_windows_temp')
HDB_POINT_TEMP_DIR = os.path.join(ARTIFACTS_DIR, 'hdb_point_temp')

print(f'Input:     {INPUT_PATH}')
print(f'Artifacts: {ARTIFACTS_DIR}')
print(f'Inputs exist:')
for path in [INPUT_PATH, SPLIT_PATH, CLIP_BOUNDS_PATH, SCALER_PATH, CONTRACT_PATH]:
    print(f'  {os.path.basename(path):>20s}: {os.path.exists(path)}')

Mounted at /content/drive
Input:     /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Artifacts: /content/drive/MyDrive/thesis_processed/models_v3_artifacts
Inputs exist:
  european_flights_annotated_v3.parquet: True
     split_metadata.json: True
        clip_bounds.json: True
           scaler.joblib: True
           contract.json: True


## 2. Load 03.1 artifacts

In [2]:
with open(CONTRACT_PATH) as f:
    contract = json.load(f)

MODEL_FEATURES = contract['model_features']
DQ_HARD = contract['dq_hard_flag']
DQ_SOFT = contract['dq_soft_flag']
FEATURE_QUALITY = contract['feature_quality_flag']
GAP_FLAG = contract['gap_flag']
RANDOM_STATE = contract['random_state']

with open(SPLIT_PATH) as f:
    split_metadata = json.load(f)

train_core_ids = np.array(sorted(split_metadata['train_core_flights']), dtype=np.int64)
calibration_val_ids = np.array(sorted(split_metadata['calibration_val_flights']), dtype=np.int64)
test_ids = np.array(sorted(split_metadata['test_flights']), dtype=np.int64)

print(f'\nSplits:')
print(f'  train_core:      {len(train_core_ids):,} flights')
print(f'  calibration_val: {len(calibration_val_ids):,} flights')
print(f'  test:            {len(test_ids):,} flights')

with open(CLIP_BOUNDS_PATH) as f:
    clip_data = json.load(f)
clip_bounds = clip_data['bounds']

scaler = joblib.load(SCALER_PATH)


Splits:
  train_core:      17,230 flights
  calibration_val:  4,307 flights
  test:             8,251 flights


In [3]:
# Конфигурация
WINDOW_LEN_SEC = 120
WINDOW_STRIDE_SEC = 60
MIN_POINTS_PER_WINDOW = 30
DQ_HARD_SHARE_TRAINING_MAX = 0.0   # 0.0 = ни одной dq_hard точки в training окне

PHASE_NAMES = {-1: 'unknown', 0: 'ground', 1: 'takeoff', 2: 'climb',
               3: 'cruise', 4: 'descent', 5: 'approach', 6: 'landing'}
PHASE_GROUP_MAP = {
    2: 'climb',
    3: 'cruise',
    4: 'descent',
    0: 'terminal', 1: 'terminal', 5: 'terminal', 6: 'terminal',
}
PHASE_GROUPS = ['climb', 'cruise', 'descent', 'terminal']

HDB_PARAMS = {
    'cruise':   {'min_cluster_size': 100, 'min_samples': 10, 'metric': 'euclidean'},
    'climb':    {'min_cluster_size': 50,  'min_samples': 10, 'metric': 'euclidean'},
    'descent':  {'min_cluster_size': 50,  'min_samples': 10, 'metric': 'euclidean'},
    'terminal': {'min_cluster_size': 30,  'min_samples': 5,  'metric': 'euclidean'},
}

PCA_COMPONENTS = 10
TRAINING_SAMPLE_PER_GROUP = 100_000
N_STABILITY_SEEDS = 3
STABILITY_SEEDS = [RANDOM_STATE, RANDOM_STATE + 1000, RANDOM_STATE + 2000]

print(f'\nWindow params:')
print(f'  window_len:          {WINDOW_LEN_SEC}s')
print(f'  window_stride:       {WINDOW_STRIDE_SEC}s (50% overlap)')
print(f'  min_points/window:   {MIN_POINTS_PER_WINDOW}')
print(f'  dq_hard training tolerance: {DQ_HARD_SHARE_TRAINING_MAX}')

print(f'\nHDBSCAN per-group params:')
for grp, params in HDB_PARAMS.items():
    print(f'  {grp:>10s}: {params}')

print(f'\nStability samples: {N_STABILITY_SEEDS} per group, seeds: {STABILITY_SEEDS}')


Window params:
  window_len:          120s
  window_stride:       60s (50% overlap)
  min_points/window:   30
  dq_hard training tolerance: 0.0

HDBSCAN per-group params:
      cruise: {'min_cluster_size': 100, 'min_samples': 10, 'metric': 'euclidean'}
       climb: {'min_cluster_size': 50, 'min_samples': 10, 'metric': 'euclidean'}
     descent: {'min_cluster_size': 50, 'min_samples': 10, 'metric': 'euclidean'}
    terminal: {'min_cluster_size': 30, 'min_samples': 5, 'metric': 'euclidean'}

Stability samples: 3 per group, seeds: [1321, 2321, 3321]


## 3. Helper: clip + scale features

In [4]:
def clip_and_scale_features(df, model_features=MODEL_FEATURES,
                            clip_bounds=clip_bounds, scaler=scaler):
    X = df[model_features].to_numpy(dtype=np.float32, copy=True)
    for i, feature in enumerate(model_features):
        bounds = clip_bounds[feature]
        np.clip(X[:, i], bounds['low'], bounds['high'], out=X[:, i])
    X_scaled = scaler.transform(X).astype(np.float32)
    for i, feature in enumerate(model_features):
        df[feature] = X_scaled[:, i]
    return df


print('clip_and_scale_features defined.')

clip_and_scale_features defined.


## 4. Pass A: построение окон с carryover для полных рейсов

Streaming через annotated_v3 с carryover-механикой:
- В каждой итерации batch комбинируется с `carry` (rows последнего рейса предыдущего batch).
- Сортируем и определяем last_fid в combined.
- Все рейсы кроме last_fid обрабатываются как complete flights.
- Last_fid идёт в carry для следующей итерации.

Это гарантирует, что ts_sec и индексы окон считаются от истинного начала рейса, а не от первой точки batch.

`row_id` присваивается до combine с carry, остаётся глобальным sequential index (consistent с if_scores.parquet).

In [5]:
print('Pass A: building windows with carryover...')
t0 = time.time()

# Чистим temp-директории
os.makedirs(HDB_WINDOWS_TEMP_DIR, exist_ok=True)
os.makedirs(HDB_POINT_TEMP_DIR, exist_ok=True)
for tmp_dir in [HDB_WINDOWS_TEMP_DIR, HDB_POINT_TEMP_DIR]:
    old = glob.glob(os.path.join(tmp_dir, '*.parquet'))
    if old:
        for f in old:
            os.remove(f)
        print(f'Removed {len(old)} stale chunks from {os.path.basename(tmp_dir)}')

read_cols = (
    ['flight_id', 'timestamp', 'flight_phase',
     DQ_HARD, DQ_SOFT, FEATURE_QUALITY, GAP_FLAG]
    + MODEL_FEATURES
)


def aggregate_windows(df_complete, window_col):
    """Aggregate window-level features for complete-flight DataFrame."""
    agg_specs = {}
    for f in MODEL_FEATURES:
        agg_specs[f] = ['mean', 'std']
    agg_specs[DQ_HARD] = 'sum'
    agg_specs[DQ_SOFT] = 'sum'
    agg_specs[FEATURE_QUALITY] = 'sum'
    agg_specs[GAP_FLAG] = 'sum'
    agg_specs['model_features_finite'] = 'sum'
    agg_specs['flight_phase'] = lambda x: x.mode().iloc[0] if len(x) > 0 else -1
    agg_specs['timestamp'] = ['min', 'max', 'count']

    agg_df = df_complete.groupby(['flight_id', window_col]).agg(agg_specs)
    agg_df.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col
                      for col in agg_df.columns]
    agg_df = agg_df.reset_index()
    return agg_df


def process_complete_flights(df_complete, batch_idx):
    """Build windows и сохраняет chunks для DataFrame содержащего ТОЛЬКО complete flights."""
    if len(df_complete) == 0:
        return 0

    # ts_sec per flight
    df_complete = df_complete.copy()
    df_complete['timestamp'] = pd.to_datetime(df_complete['timestamp'])
    df_complete['ts_sec'] = df_complete.groupby('flight_id')['timestamp'].transform(
        lambda x: ((x - x.iloc[0]).dt.total_seconds()).astype('int64')
    )

    # Индексы окон
    df_complete['window_a'] = (df_complete['ts_sec'] // WINDOW_LEN_SEC).astype('int64')
    df_complete['window_b'] = (
        (df_complete['ts_sec'] - WINDOW_STRIDE_SEC) // WINDOW_LEN_SEC
    ).astype('int64')

    # Point-level info (для последующего window-to-point map)
    fid = df_complete['flight_id'].to_numpy()
    is_calval = np.isin(fid, calibration_val_ids)
    is_test = np.isin(fid, test_ids)
    is_train_core = np.isin(fid, train_core_ids)
    split_arr = np.full(len(df_complete), 'unknown', dtype=object)
    split_arr[is_train_core] = 'train_core'
    split_arr[is_calval] = 'calibration_val'
    split_arr[is_test] = 'test'

    point_cols = ['row_id', 'flight_id', 'timestamp', 'flight_phase',
                  DQ_HARD, DQ_SOFT, FEATURE_QUALITY,
                  'window_a', 'window_b']
    df_pts = df_complete[point_cols].copy()
    df_pts['split'] = split_arr

    pt_path = os.path.join(HDB_POINT_TEMP_DIR, f'point_chunk_{batch_idx:03d}.parquet')
    df_pts.to_parquet(pt_path, index=False, compression='snappy')

    # Aggregate windows (два прохода: window_a, window_b)
    agg_a = aggregate_windows(df_complete, 'window_a')
    agg_a = agg_a.rename(columns={'window_a': 'window_idx'})
    agg_a['window_offset'] = 0

    df_b = df_complete[df_complete['window_b'] >= 0]
    if len(df_b) > 0:
        agg_b = aggregate_windows(df_b, 'window_b')
        agg_b = agg_b.rename(columns={'window_b': 'window_idx'})
        agg_b['window_offset'] = 1
        agg = pd.concat([agg_a, agg_b], ignore_index=True)
    else:
        agg = agg_a

    # Фильтр min points
    n_points_col = 'timestamp_count'
    agg = agg[agg[n_points_col] >= MIN_POINTS_PER_WINDOW].copy()

    # Honest finite_share: доля точек со всеми 12 features finite
    agg['finite_share'] = (
        agg['model_features_finite_sum'] / agg[n_points_col]
    ).astype('float32')
    agg['window_finite'] = (agg['finite_share'] == 1.0).astype('uint8')

    # DQ shares
    agg['dq_hard_share'] = (
        agg[f'{DQ_HARD}_sum'] / agg[n_points_col]).astype('float32')
    agg['dq_soft_share'] = (
        agg[f'{DQ_SOFT}_sum'] / agg[n_points_col]).astype('float32')
    agg['feature_quality_share'] = (
        agg[f'{FEATURE_QUALITY}_sum'] / agg[n_points_col]).astype('float32')
    agg['gap_share'] = (
        agg[f'{GAP_FLAG}_sum'] / agg[n_points_col]).astype('float32')

    # Dominant phase
    agg = agg.rename(columns={'flight_phase_<lambda>': 'dominant_phase'})

    # NaN std (single-point windows) заменяем на 0
    for f in MODEL_FEATURES:
        agg[f'{f}_std'] = agg[f'{f}_std'].fillna(0).astype('float32')
        agg[f'{f}_mean'] = agg[f'{f}_mean'].astype('float32')

    # Phase group
    agg['phase_group'] = agg['dominant_phase'].map(PHASE_GROUP_MAP).fillna('unknown')

    # Split
    fid_w = agg['flight_id'].to_numpy()
    is_train_core_w = np.isin(fid_w, train_core_ids)
    is_calval_w = np.isin(fid_w, calibration_val_ids)
    is_test_w = np.isin(fid_w, test_ids)
    split_w = np.full(len(agg), 'unknown', dtype=object)
    split_w[is_train_core_w] = 'train_core'
    split_w[is_calval_w] = 'calibration_val'
    split_w[is_test_w] = 'test'
    agg['split'] = split_w

    # Drop intermediate sum cols
    drop_cols = [f'{DQ_HARD}_sum', f'{DQ_SOFT}_sum',
                 f'{FEATURE_QUALITY}_sum', f'{GAP_FLAG}_sum',
                 'model_features_finite_sum']
    agg = agg.drop(columns=drop_cols, errors='ignore')

    chunk_path = os.path.join(HDB_WINDOWS_TEMP_DIR,
                              f'windows_chunk_{batch_idx:03d}.parquet')
    agg.to_parquet(chunk_path, index=False, compression='snappy')

    return len(agg)

# Streaming pass с carryover
pf = pq.ParquetFile(INPUT_PATH)
batch_size = 5_000_000

carry = pd.DataFrame()
global_offset = 0
n_batches = 0
n_windows_total = 0

for batch in pf.iter_batches(batch_size=batch_size, columns=read_cols):
    df_new = batch.to_pandas()
    n_new = len(df_new)
    n_batches += 1

    # row_id присваивается ДО combine с carry, то есть глобальный sequential
    df_new['row_id'] = np.arange(global_offset, global_offset + n_new, dtype=np.int64)
    global_offset += n_new

    # Применяем clip + scale к df_new
    df_new = clip_and_scale_features(df_new)

    # finite-индикатор (per-row)
    df_new['model_features_finite'] = df_new[MODEL_FEATURES].notna().all(axis=1).astype('uint8')

    # Объединяем с carry
    if len(carry) > 0:
        df_combined = pd.concat([carry, df_new], ignore_index=True)
    else:
        df_combined = df_new

    df_combined = df_combined.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)

    # Последний рейс уходит в carry
    last_fid = df_combined['flight_id'].iloc[-1]
    last_mask = df_combined['flight_id'] == last_fid
    process_df = df_combined[~last_mask].copy()
    carry = df_combined[last_mask].copy()

    if len(process_df) > 0:
        n_w_saved = process_complete_flights(process_df, n_batches - 1)
        n_windows_total += n_w_saved

    print(f'  batch {n_batches}: rows={n_new:,}, '
          f'carry={len(carry):,}, windows so far={n_windows_total:,}',
          end='\r')

    del df_new, df_combined, process_df

# Final carry processing
if len(carry) > 0:
    n_w_saved = process_complete_flights(carry, n_batches)
    n_windows_total += n_w_saved
    print(f'\n  Final carry processed: {len(carry):,} rows, +{n_w_saved:,} windows')

del carry

print()
print(f'\nPass A complete: {time.time() - t0:.0f}s')
print(f'Total windows: {n_windows_total:,}')

Pass A: building windows with carryover...
  batch 30: rows=4,129,454, carry=8,738, windows so far=2,505,124
  Final carry processed: 8,738 rows, +149 windows


Pass A complete: 691s
Total windows: 2,505,273


## 5. Слияние окон, sanity-check и breakdown

После Pass A: загружаем все window chunks, проверяем уникальность ключей окна (это гарантия корректности carryover), делаем breakdown по фазам и splits.

In [6]:
print('Loading and combining window chunks...')
t0 = time.time()

window_chunks = sorted(glob.glob(os.path.join(HDB_WINDOWS_TEMP_DIR, '*.parquet')))
all_windows = []
for cp in window_chunks:
    all_windows.append(pd.read_parquet(cp))
windows = pd.concat(all_windows, ignore_index=True)
del all_windows

print(f'Total windows in memory: {len(windows):,}')
print(f'Memory: {windows.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# Sanity: дубликатов window keys быть не должно (carryover correctness)
duplicates = windows.duplicated(
    subset=['flight_id', 'window_idx', 'window_offset']
).sum()
assert duplicates == 0, (
    f'{duplicates:,} duplicate (flight_id, window_idx, window_offset) keys, '
    f'carryover broken! Pass A нужно отлаживать.'
)
print('Sanity: window keys unique')

# Глобальный window_id (после concat, carryover сохраняет порядок)
windows = windows.sort_values(['flight_id', 'window_idx', 'window_offset']).reset_index(drop=True)
windows['window_id'] = np.arange(len(windows), dtype=np.int64)

print(f'\nWindows by phase group:')
for grp in PHASE_GROUPS + ['unknown']:
    n = (windows['phase_group'] == grp).sum()
    print(f'  {grp:>10s}: {n:,} ({n / len(windows) * 100:.2f}%)')

print(f'\nWindows by split:')
for sp in ['train_core', 'calibration_val', 'test', 'unknown']:
    n = (windows['split'] == sp).sum()
    print(f'  {sp:>16s}: {n:,}')

print(f'\nWindow finite share:')
n_finite = (windows['window_finite'] == 1).sum()
print(f'  finite (all 12 features per point all finite): '
      f'{n_finite:,} ({n_finite / len(windows) * 100:.2f}%)')

print(f'\nClean training windows per group '
      f'(train_core, finite, no DQ shares, no gaps):')
for grp in PHASE_GROUPS:
    grp_train = windows[
        (windows['phase_group'] == grp)
        & (windows['split'] == 'train_core')
        & (windows['window_finite'] == 1)
        & (windows['dq_hard_share'] == 0.0)
        & (windows['dq_soft_share'] == 0)
        & (windows['feature_quality_share'] == 0)
        & (windows['gap_share'] == 0)
    ]
    print(f'  {grp:>10s}: {len(grp_train):,}')

print(f'\nLoad complete: {time.time() - t0:.0f}s')

Loading and combining window chunks...
Total windows in memory: 2,505,273
Memory: 0.70 GB
Sanity: window keys unique ✓

Windows by phase group:
       climb:    466,518 (18.62%)
      cruise:  1,264,033 (50.45%)
     descent:    441,380 (17.62%)
    terminal:    318,809 (12.73%)
     unknown:     14,533 (0.58%)

Windows by split:
        train_core:  1,446,172
   calibration_val:    361,496
              test:    697,605
           unknown:          0

Window finite share:
  finite (all 12 features per point all finite): 2,433,347 (97.13%)

Clean training windows per group (train_core, finite, no DQ shares, no gaps):
       climb:    229,242
      cruise:    676,816
     descent:    215,789
    terminal:    155,451

Load complete: 8s


## 6. Train HDBSCAN per phase group (3 stability samples)

Per group:
1. Filter clean training pool (train_core, finite, no DQ).
2. 3 random samples по `TRAINING_SAMPLE_PER_GROUP` windows.
3. Per-(group, seed): fit StandardScaler (24-dim), потом PCA, потом HDBSCAN.
4. Save artifacts.

In [7]:
print('Training HDBSCAN per phase group...')
t0_train = time.time()

window_feature_cols = (
    [f'{f}_mean' for f in MODEL_FEATURES]
    + [f'{f}_std' for f in MODEL_FEATURES]
)
print(f'Window feature columns: {len(window_feature_cols)} (12 means + 12 stds)')

hdb_models = {}  # (group, seed_idx) в dict

for grp in PHASE_GROUPS:
    print(f'\n=== Group: {grp} ===')

    train_pool = windows[
        (windows['phase_group'] == grp)
        & (windows['split'] == 'train_core')
        & (windows['window_finite'] == 1)
        & (windows['dq_hard_share'] == 0.0)
        & (windows['dq_soft_share'] == 0)
        & (windows['feature_quality_share'] == 0)
        & (windows['gap_share'] == 0)
    ]
    n_train_pool = len(train_pool)
    print(f'  Clean training pool: {n_train_pool:,}')

    if n_train_pool < 1000:
        print(f'  WARN: too few training windows ({n_train_pool}), skipping group')
        continue

    sample_size = min(TRAINING_SAMPLE_PER_GROUP, n_train_pool)

    for seed_idx, seed in enumerate(STABILITY_SEEDS):
        seed_label = 'main' if seed_idx == 0 else f's{seed_idx + 1}'
        rng = np.random.RandomState(seed)
        sample_idx = rng.choice(n_train_pool, size=sample_size, replace=False)
        X_train = train_pool[window_feature_cols].to_numpy()[sample_idx]

        grp_scaler = StandardScaler()
        X_train_scaled = grp_scaler.fit_transform(X_train).astype(np.float32)

        n_comp = min(PCA_COMPONENTS, X_train_scaled.shape[1])
        pca = PCA(n_components=n_comp, random_state=seed)
        X_train_pca = pca.fit_transform(X_train_scaled).astype(np.float32)
        explained = pca.explained_variance_ratio_.sum()
        print(f'  [{seed_label}] sample {sample_size:,}, PCA {n_comp} comp '
              f'({explained:.3f} var)', end='', flush=True)

        params = HDB_PARAMS[grp]
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=params['min_cluster_size'],
            min_samples=params['min_samples'],
            metric=params['metric'],
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1,
        )
        t1 = time.time()
        clusterer.fit(X_train_pca)
        n_clusters = len(np.unique(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
        n_noise = int((clusterer.labels_ == -1).sum())
        print(f', HDBSCAN {time.time() - t1:.0f}s, {n_clusters} clusters, '
              f'{n_noise:,} noise ({n_noise / sample_size * 100:.1f}%)')

        hdb_models[(grp, seed_idx)] = {
            'scaler': grp_scaler,
            'pca': pca,
            'hdb': clusterer,
            'sample_size': sample_size,
            'n_clusters': n_clusters,
            'noise_share': n_noise / sample_size,
        }

        joblib.dump(grp_scaler,
                    os.path.join(ARTIFACTS_DIR, f'hdb_scaler_{grp}_{seed_label}.joblib'))
        joblib.dump(pca,
                    os.path.join(ARTIFACTS_DIR, f'hdb_pca_{grp}_{seed_label}.joblib'))
        joblib.dump(clusterer,
                    os.path.join(ARTIFACTS_DIR, f'hdb_{grp}_{seed_label}.joblib'))

    del train_pool

print(f'\nHDBSCAN training complete: {time.time() - t0_train:.0f}s')

Training HDBSCAN per phase group...
Window feature columns: 24 (12 means + 12 stds)

=== Group: climb ===
  Clean training pool: 229,242
  [main] sample 100,000, PCA 10 comp (0.873 var) → HDBSCAN 57s, 2 clusters, 26,746 noise (26.7%)
  [s2] sample 100,000, PCA 10 comp (0.873 var) → HDBSCAN 50s, 2 clusters, 22,572 noise (22.6%)
  [s3] sample 100,000, PCA 10 comp (0.873 var) → HDBSCAN 51s, 5 clusters, 58,608 noise (58.6%)

=== Group: cruise ===
  Clean training pool: 676,816
  [main] sample 100,000, PCA 10 comp (0.861 var) → HDBSCAN 48s, 3 clusters, 21,307 noise (21.3%)
  [s2] sample 100,000, PCA 10 comp (0.862 var) → HDBSCAN 50s, 4 clusters, 19,878 noise (19.9%)
  [s3] sample 100,000, PCA 10 comp (0.863 var) → HDBSCAN 53s, 3 clusters, 22,367 noise (22.4%)

=== Group: descent ===
  Clean training pool: 215,789
  [main] sample 100,000, PCA 10 comp (0.842 var) → HDBSCAN 58s, 2 clusters, 66,554 noise (66.6%)
  [s2] sample 100,000, PCA 10 comp (0.841 var) → HDBSCAN 62s, 2 clusters, 25,713 no

## 7. Score windows per group

Main: ВСЕ window_finite windows (всех splits).
Stability seeds: только calibration_val (для stability analysis).

`approximate_predict_scores` возвращает один массив, clip(0, 1) для numerical safety.

In [8]:
print('Scoring windows...')
t0 = time.time()

windows['hdb_score_raw_main'] = np.nan
windows['hdb_score_raw_s2'] = np.nan
windows['hdb_score_raw_s3'] = np.nan

for grp in PHASE_GROUPS:
    if (grp, 0) not in hdb_models:
        continue

    grp_mask = (windows['phase_group'] == grp) & (windows['window_finite'] == 1)
    n_grp = int(grp_mask.sum())
    if n_grp == 0:
        continue

    print(f'\n  Group {grp}: {n_grp:,} eligible windows')

    X = windows.loc[grp_mask, window_feature_cols].to_numpy(dtype=np.float32)

    # Main: all eligible
    m_main = hdb_models[(grp, 0)]
    X_scaled = m_main['scaler'].transform(X).astype(np.float32)
    X_pca = m_main['pca'].transform(X_scaled).astype(np.float32)
    scores_main = hdbscan.approximate_predict_scores(m_main['hdb'], X_pca)
    scores_main = np.clip(scores_main, 0, 1).astype(np.float32)
    windows.loc[grp_mask, 'hdb_score_raw_main'] = scores_main
    print(f'    main: scored, range [{scores_main.min():.2f}, {scores_main.max():.2f}]')
    del X_scaled, X_pca, scores_main

    # Stability seeds: только calibration_val
    calval_mask_grp = grp_mask & (windows['split'] == 'calibration_val')
    n_calval_grp = int(calval_mask_grp.sum())
    if n_calval_grp > 0:
        X_calval = windows.loc[calval_mask_grp, window_feature_cols].to_numpy(dtype=np.float32)
        for seed_idx in [1, 2]:
            m = hdb_models[(grp, seed_idx)]
            seed_label = f's{seed_idx + 1}'
            X_cs = m['scaler'].transform(X_calval).astype(np.float32)
            X_cp = m['pca'].transform(X_cs).astype(np.float32)
            scores_calval = hdbscan.approximate_predict_scores(m['hdb'], X_cp)
            scores_calval = np.clip(scores_calval, 0, 1).astype(np.float32)
            col = f'hdb_score_raw_{seed_label}'
            windows.loc[calval_mask_grp, col] = scores_calval
            print(f'    {seed_label}: calval scored, range '
                  f'[{scores_calval.min():.2f}, {scores_calval.max():.2f}]')
            del X_cs, X_cp, scores_calval
        del X_calval

    del X

print(f'\nScoring complete: {time.time() - t0:.0f}s')
n_scored = int(windows['hdb_score_raw_main'].notna().sum())
print(f'Windows with hdb_score_raw_main: {n_scored:,}')

Scoring windows...

  Group climb: 438,228 eligible windows
    main: scored, range [0.0000, 0.9956]
    s2: calval scored, range [0.0000, 0.9937]
    s3: calval scored, range [0.0000, 0.9941]

  Group cruise: 1,252,100 eligible windows
    main: scored, range [0.0000, 0.9990]
    s2: calval scored, range [0.0000, 0.9989]
    s3: calval scored, range [0.0000, 0.9989]

  Group descent: 437,127 eligible windows
    main: scored, range [0.0000, 0.9935]
    s2: calval scored, range [0.0000, 0.9931]
    s3: calval scored, range [0.0000, 0.9922]

  Group terminal: 305,892 eligible windows
    main: scored, range [0.0000, 0.9985]
    s2: calval scored, range [0.0000, 0.9985]
    s3: calval scored, range [0.0000, 0.9984]

Scoring complete: 1155s
Windows with hdb_score_raw_main: 2,433,347


## 8. Window-level percentile vs clean calibration_val

ECDF percentile against clean calibration windows:
- `hdb_score_global_pct`: across all clean cal windows.
- `hdb_score_phase_pct`: within phase_group, against same-group clean cal.

In [9]:
print('Computing window-level percentiles vs clean calibration_val...')

clean_ref_mask = (
    (windows['split'] == 'calibration_val')
    & (windows['window_finite'] == 1)
    & (windows['dq_hard_share'] == 0.0)
    & (windows['dq_soft_share'] == 0)
    & (windows['feature_quality_share'] == 0)
    & (windows['gap_share'] == 0)
    & (windows['hdb_score_raw_main'].notna())
)
n_ref = int(clean_ref_mask.sum())
print(f'Clean calibration windows for reference: {n_ref:,}')

ref_global_sorted = np.sort(windows.loc[clean_ref_mask, 'hdb_score_raw_main'].to_numpy())
print(f'  Global ref range: [{ref_global_sorted[0]:.2f}, {ref_global_sorted[-1]:.2f}]')
print(f'  P50/P90/P99/P99.9: '
      f'{np.quantile(ref_global_sorted, [0.5, 0.9, 0.99, 0.999]).round(4)}')

# Global pct
windows['hdb_score_global_pct'] = np.nan
finite_main = windows['hdb_score_raw_main'].notna()
windows.loc[finite_main, 'hdb_score_global_pct'] = (
    np.searchsorted(ref_global_sorted,
                    windows.loc[finite_main, 'hdb_score_raw_main'].to_numpy(),
                    side='right') / len(ref_global_sorted)
).astype('float32')

# Phase-group pct
windows['hdb_score_phase_pct'] = np.nan
n_phase_fallback = 0
for grp in PHASE_GROUPS:
    grp_clean_mask = clean_ref_mask & (windows['phase_group'] == grp)
    grp_ref = np.sort(windows.loc[grp_clean_mask, 'hdb_score_raw_main'].to_numpy())
    grp_finite_mask = finite_main & (windows['phase_group'] == grp)

    if len(grp_ref) < 100:
        print(f'  WARN {grp}: only {len(grp_ref)} clean cal ref, fallback to global')
        windows.loc[grp_finite_mask, 'hdb_score_phase_pct'] = windows.loc[
            grp_finite_mask, 'hdb_score_global_pct']
        n_phase_fallback += int(grp_finite_mask.sum())
        continue

    windows.loc[grp_finite_mask, 'hdb_score_phase_pct'] = (
        np.searchsorted(grp_ref,
                        windows.loc[grp_finite_mask, 'hdb_score_raw_main'].to_numpy(),
                        side='right') / len(grp_ref)
    ).astype('float32')
    print(f'  {grp:>10s}: ref n={len(grp_ref):,}, scored {int(grp_finite_mask.sum()):,}')

print(f'\nPhase pct fallback to global: {n_phase_fallback:,}')

# Сохраняем window-level scores
print(f'\nSaving window-level scores to {HDB_WINDOW_SCORES_PATH}')
keep_cols = [
    'window_id', 'flight_id', 'dominant_phase', 'phase_group', 'split',
    'window_offset', 'window_idx', 'window_finite', 'finite_share',
    'timestamp_min', 'timestamp_max', 'timestamp_count',
    'dq_hard_share', 'dq_soft_share', 'feature_quality_share', 'gap_share',
    'hdb_score_raw_main', 'hdb_score_raw_s2', 'hdb_score_raw_s3',
    'hdb_score_global_pct', 'hdb_score_phase_pct',
]
keep_cols = [c for c in keep_cols if c in windows.columns]
windows[keep_cols].to_parquet(HDB_WINDOW_SCORES_PATH, index=False, compression='snappy')
print(f'  Size: {os.path.getsize(HDB_WINDOW_SCORES_PATH) / 1e6:.1f} MB')

Computing window-level percentiles vs clean calibration_val...
Clean calibration windows for reference: 318,049
  Global ref range: [0.0000, 0.9934]
  P50/P90/P99/P99.9: [0.6517 0.9337 0.9634 0.9762]
       climb: ref n=56,167, scored 438,228
      cruise: ref n=169,603, scored 1,252,100
     descent: ref n=53,379, scored 437,127
    terminal: ref n=38,900, scored 305,892

Phase pct fallback to global: 0

Saving window-level scores → /content/drive/MyDrive/thesis_processed/models_v3_artifacts/hdb_window_scores.parquet
  Size: 88.5 MB


## 9. Map window scores to point scores (через offset key)

Каждая точка попадает в 0/1/2 окна (window_a с offset=0, window_b с offset=1). Lookup score через явный (flight_id, window_idx, window_offset) key, без post-filter, без cartesian risk.

Point score = max(score_a, score_b) (np.fmax, NaN-aware).

In [10]:
print('Mapping window scores to point scores...')
t0 = time.time()

# Lookup-таблица
win_lookup = windows[['flight_id', 'window_idx', 'window_offset',
                      'hdb_score_raw_main', 'hdb_score_raw_s2', 'hdb_score_raw_s3',
                      'hdb_score_global_pct', 'hdb_score_phase_pct']].copy()

point_chunks = sorted(glob.glob(os.path.join(HDB_POINT_TEMP_DIR, '*.parquet')))
print(f'Processing {len(point_chunks)} point chunks...')

writer = None
n_points_written = 0

for cp in point_chunks:
    pts = pd.read_parquet(cp)

    # Оставляем только calval + test (train_core points не нужны на point-level)
    pts = pts[pts['split'].isin(['calibration_val', 'test'])].copy()
    # Merge for window_a (offset=0), explicit offset in key
    pts['window_offset'] = 0
    merged_a = pts.merge(
        win_lookup.rename(columns={'window_idx': 'window_a'}),
        on=['flight_id', 'window_a', 'window_offset'],
        how='left',
    )
    score_main_a = merged_a['hdb_score_raw_main'].to_numpy()
    score_s2_a = merged_a['hdb_score_raw_s2'].to_numpy()
    score_s3_a = merged_a['hdb_score_raw_s3'].to_numpy()
    score_glob_a = merged_a['hdb_score_global_pct'].to_numpy()
    score_phase_a = merged_a['hdb_score_phase_pct'].to_numpy()
    del merged_a

    # Merge for window_b (offset=1)
    pts['window_offset'] = 1
    merged_b = pts.merge(
        win_lookup.rename(columns={'window_idx': 'window_b'}),
        on=['flight_id', 'window_b', 'window_offset'],
        how='left',
    )
    score_main_b = merged_b['hdb_score_raw_main'].to_numpy()
    score_s2_b = merged_b['hdb_score_raw_s2'].to_numpy()
    score_s3_b = merged_b['hdb_score_raw_s3'].to_numpy()
    score_glob_b = merged_b['hdb_score_global_pct'].to_numpy()
    score_phase_b = merged_b['hdb_score_phase_pct'].to_numpy()
    del merged_b

    # Удаляем вспомогательную колонку
    pts = pts.drop(columns=['window_offset'])

    # Берём max (NaN-aware)
    pts['hdb_score_raw_main'] = np.fmax(score_main_a, score_main_b).astype('float32')
    pts['hdb_score_raw_s2'] = np.fmax(score_s2_a, score_s2_b).astype('float32')
    pts['hdb_score_raw_s3'] = np.fmax(score_s3_a, score_s3_b).astype('float32')
    pts['hdb_score_global_pct'] = np.fmax(score_glob_a, score_glob_b).astype('float32')
    pts['hdb_score_phase_pct'] = np.fmax(score_phase_a, score_phase_b).astype('float32')

    # is_clean_reference
    pts['is_clean_reference'] = (
        (pts[DQ_HARD] == 0)
        & (pts[DQ_SOFT] == 0)
        & (pts[FEATURE_QUALITY] == 0)
    ).astype('uint8')

    out_cols = [
        'row_id', 'flight_id', 'timestamp', 'flight_phase', 'split',
        DQ_HARD, DQ_SOFT, FEATURE_QUALITY, 'is_clean_reference',
        'hdb_score_raw_main', 'hdb_score_raw_s2', 'hdb_score_raw_s3',
        'hdb_score_global_pct', 'hdb_score_phase_pct',
    ]
    pts_out = pts[out_cols].copy()
    pts_out[DQ_HARD] = pts_out[DQ_HARD].astype('uint8')
    pts_out[DQ_SOFT] = pts_out[DQ_SOFT].astype('uint8')
    pts_out[FEATURE_QUALITY] = pts_out[FEATURE_QUALITY].astype('uint8')
    pts_out['flight_phase'] = pts_out['flight_phase'].astype('int8')
    pts_out['split'] = pd.Categorical(
        pts_out['split'],
        categories=['calibration_val', 'test'])

    table = pa.Table.from_pandas(pts_out, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(HDB_POINT_SCORES_PATH, table.schema,
                                   compression='snappy')
    writer.write_table(table)
    n_points_written += len(pts_out)

    print(f'  {os.path.basename(cp)}: {len(pts_out):,} written, '
          f'total={n_points_written:,}', end='\r')

    del pts, pts_out, table
    del score_main_a, score_main_b, score_s2_a, score_s2_b
    del score_s3_a, score_s3_b, score_glob_a, score_glob_b
    del score_phase_a, score_phase_b

if writer is not None:
    writer.close()
print()
print(f'\nWindow-to-point mapping complete: {time.time() - t0:.0f}s')
print(f'Total points written: {n_points_written:,}')
print(f'File size: {os.path.getsize(HDB_POINT_SCORES_PATH) / 1e6:.1f} MB')

Mapping window scores → point scores...
Processing 31 point chunks...
  point_chunk_030.parquet: 8,738 written, total=63,019,803

Window→point mapping complete: 146s
Total points written: 63,019,803
File size: 428.6 MB


In [11]:
# Cleanup temp dirs
for cp in window_chunks:
    if os.path.exists(cp):
        os.remove(cp)
for cp in point_chunks:
    if os.path.exists(cp):
        os.remove(cp)
try:
    os.rmdir(HDB_WINDOWS_TEMP_DIR)
    os.rmdir(HDB_POINT_TEMP_DIR)
    print('Removed temp directories.')
except OSError:
    pass

del win_lookup

Removed temp directories.


0

## 10. Stability analysis на уровне окон

Для top-k используем `np.argpartition` вместо полной `np.argsort`, это заметно быстрее на больших массивах.

In [12]:
print('Pass C: stability analysis on calibration_val windows...')

calval_w = windows[
    (windows['split'] == 'calibration_val')
    & (windows['hdb_score_raw_main'].notna())
    & (windows['hdb_score_raw_s2'].notna())
    & (windows['hdb_score_raw_s3'].notna())
].copy()

n_all = len(calval_w)
clean_w_mask = (
    (calval_w['dq_hard_share'] == 0.0)
    & (calval_w['dq_soft_share'] == 0)
    & (calval_w['feature_quality_share'] == 0)
    & (calval_w['gap_share'] == 0)
)
n_clean = int(clean_w_mask.sum())

print(f'\nCalval windows with all 3 seed scores: {n_all:,}')
print(f'  Clean subset: {n_clean:,} ({n_clean / n_all * 100:.2f}%)')


def jaccard_top_pct(scores_a, scores_b, top_pct=0.01):
    """Jaccard between top-N indices via argpartition (faster than argsort)."""
    n = len(scores_a)
    n_top = max(1, int(n * top_pct))
    if n_top >= n:
        return 1.0
    top_a = set(np.argpartition(scores_a, -n_top)[-n_top:])
    top_b = set(np.argpartition(scores_b, -n_top)[-n_top:])
    inter = len(top_a & top_b)
    union = len(top_a | top_b)
    return inter / union if union > 0 else 0.0


def compute_stability_metrics(s_main, s_2, s_3):
    return {
        'n_points': len(s_main),
        'jaccard_top_1pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.01)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.01)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.01)),
        },
        'jaccard_top_0_5pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.005)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.005)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.005)),
        },
        'jaccard_top_0_1pct': {
            'main_vs_s2': float(jaccard_top_pct(s_main, s_2, 0.001)),
            'main_vs_s3': float(jaccard_top_pct(s_main, s_3, 0.001)),
            's2_vs_s3': float(jaccard_top_pct(s_2, s_3, 0.001)),
        },
        'pearson': {
            'main_vs_s2': float(np.corrcoef(s_main, s_2)[0, 1]),
            'main_vs_s3': float(np.corrcoef(s_main, s_3)[0, 1]),
            's2_vs_s3': float(np.corrcoef(s_2, s_3)[0, 1]),
        }
    }


s_main_all = calval_w['hdb_score_raw_main'].to_numpy()
s_2_all = calval_w['hdb_score_raw_s2'].to_numpy()
s_3_all = calval_w['hdb_score_raw_s3'].to_numpy()
stab_all = compute_stability_metrics(s_main_all, s_2_all, s_3_all)

clean_w = calval_w[clean_w_mask]
s_main_c = clean_w['hdb_score_raw_main'].to_numpy()
s_2_c = clean_w['hdb_score_raw_s2'].to_numpy()
s_3_c = clean_w['hdb_score_raw_s3'].to_numpy()
stab_clean = compute_stability_metrics(s_main_c, s_2_c, s_3_c)

del calval_w, clean_w

Pass C: stability analysis on calibration_val windows...

Calval windows with all 3 seed scores: 351,287
  Clean subset: 318,049 (90.54%)


0

In [13]:
stability = {
    'level': 'window',
    'all_calibration_val': stab_all,
    'clean_calibration_val': stab_clean,
}

print(f'Stability on ALL calval windows')
for metric_name in ['jaccard_top_1pct', 'jaccard_top_0_5pct',
                    'jaccard_top_0_1pct', 'pearson']:
    print(f'{metric_name}:')
    for k, v in stab_all[metric_name].items():
        print(f'  {k}: {v:.2f}')

print(f'\n=== Stability on CLEAN calval windows ===')
for metric_name in ['jaccard_top_1pct', 'jaccard_top_0_5pct',
                    'jaccard_top_0_1pct', 'pearson']:
    print(f'{metric_name}:')
    for k, v in stab_clean[metric_name].items():
        print(f'  {k}: {v:.2f}')

mean_jaccard_clean_1pct = np.mean(list(stab_clean['jaccard_top_1pct'].values()))
print(f'\nMean Jaccard top-1% on clean: {mean_jaccard_clean_1pct:.2f}')
if mean_jaccard_clean_1pct > 0.6:
    print('  Clean stability: STABLE')
elif mean_jaccard_clean_1pct > 0.4:
    print('  Clean stability: MODERATE, document as known limitation.')
else:
    print('  Clean stability: WEAK, phase-aware HDBSCAN sensitive to sample.')

with open(HDB_STABILITY_PATH, 'w') as f:
    json.dump(stability, f, indent=2)
print(f'\nSaved: {HDB_STABILITY_PATH}')

=== Stability on ALL calval windows ===
jaccard_top_1pct:
  main_vs_s2: 0.8731
  main_vs_s3: 0.9113
  s2_vs_s3: 0.8741
jaccard_top_0_5pct:
  main_vs_s2: 0.9631
  main_vs_s3: 0.9708
  s2_vs_s3: 0.9587
jaccard_top_0_1pct:
  main_vs_s2: 0.9719
  main_vs_s3: 0.9719
  s2_vs_s3: 0.9887
pearson:
  main_vs_s2: 0.9432
  main_vs_s3: 0.9497
  s2_vs_s3: 0.9342

=== Stability on CLEAN calval windows ===
jaccard_top_1pct:
  main_vs_s2: 0.7278
  main_vs_s3: 0.8344
  s2_vs_s3: 0.7555
jaccard_top_0_5pct:
  main_vs_s2: 0.7161
  main_vs_s3: 0.8182
  s2_vs_s3: 0.7434
jaccard_top_0_1pct:
  main_vs_s2: 0.7425
  main_vs_s3: 0.8223
  s2_vs_s3: 0.7716
pearson:
  main_vs_s2: 0.9415
  main_vs_s3: 0.9485
  s2_vs_s3: 0.9323

Mean Jaccard top-1% on clean: 0.7726
  Clean stability: STABLE

Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/hdb_stability.json


## 11. Top-10 anomaly preview (streaming top-k)

Streaming через hdb_scores.parquet, без чтения всего файла в RAM. Поддерживаем 3 running top-N: overall, clean, tainted.

In [14]:
print('\nTop-10 anomaly preview (streaming top-k)...')

N_TOP = 10
top_overall = pd.DataFrame()
top_clean = pd.DataFrame()
top_tainted = pd.DataFrame()

cols_for_preview = ['flight_id', 'split', 'flight_phase',
                    'hdb_score_raw_main', 'hdb_score_global_pct', 'hdb_score_phase_pct',
                    DQ_HARD, DQ_SOFT, FEATURE_QUALITY, 'is_clean_reference']

pf_scores = pq.ParquetFile(HDB_POINT_SCORES_PATH)

for batch in pf_scores.iter_batches(batch_size=2_000_000, columns=cols_for_preview):
    df = batch.to_pandas().dropna(subset=['hdb_score_raw_main'])
    # Overall
    candidates = pd.concat([top_overall, df]) if len(top_overall) else df
    top_overall = candidates.nlargest(N_TOP, 'hdb_score_raw_main')

    # Clean
    df_clean = df[df['is_clean_reference'] == 1]
    if len(df_clean) > 0:
        candidates_c = pd.concat([top_clean, df_clean]) if len(top_clean) else df_clean
        top_clean = candidates_c.nlargest(N_TOP, 'hdb_score_raw_main')

    # Tainted
    df_tainted = df[(df[DQ_HARD] == 1) | (df[FEATURE_QUALITY] == 1)]
    if len(df_tainted) > 0:
        candidates_t = pd.concat([top_tainted, df_tainted]) if len(top_tainted) else df_tainted
        top_tainted = candidates_t.nlargest(N_TOP, 'hdb_score_raw_main')


def _print_top(df_top, title, full=False):
    print(f'\n{title}:')
    if len(df_top) == 0:
        print('  (no points)')
        return
    if full:
        cols = ['flight_id', 'split', 'phase_name', 'hdb_score_raw_main',
                'hdb_score_global_pct', 'hdb_score_phase_pct',
                DQ_HARD, FEATURE_QUALITY]
    else:
        cols = ['flight_id', 'split', 'phase_name', 'hdb_score_raw_main',
                'hdb_score_global_pct', 'hdb_score_phase_pct']
    df_top = df_top.copy()
    df_top['phase_name'] = df_top['flight_phase'].map(
        lambda p: PHASE_NAMES.get(p, 'unknown')
    )
    print(df_top[cols].to_string(index=False))


_print_top(top_overall, 'Top-10 overall by hdb_score_raw_main', full=True)
_print_top(top_clean, 'Top-10 CLEAN (potential operational anomalies)')
_print_top(top_tainted, 'Top-10 DQ-TAINTED (likely artifacts)', full=True)

del top_overall, top_clean, top_tainted


Top-10 anomaly preview (streaming top-k)...

--- Top-10 overall by hdb_score_raw_main ---
 flight_id split phase_name  hdb_score_raw_main  hdb_score_global_pct  hdb_score_phase_pct  dq_hard_flag  feature_quality_flag
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    landing            0.998943                   1.0                  1.0             0                     0
 256926982  test    

0

## 12. Summary

In [15]:
print('\n' + '=' * 60)
print('03.3 SUMMARY')
print('=' * 60)

print(f'\nWindows total: {len(windows):,}')
print(f'Per phase group:')
for grp in PHASE_GROUPS:
    n_grp = (windows['phase_group'] == grp).sum()
    n_grp_finite = ((windows['phase_group'] == grp)
                    & (windows['window_finite'] == 1)).sum()
    print(f'  {grp:>10s}: {n_grp:,} windows ({n_grp_finite:,} finite)')

print(f'\nHDBSCAN models trained:')
for grp in PHASE_GROUPS:
    if (grp, 0) in hdb_models:
        m = hdb_models[(grp, 0)]
        print(f'  {grp:>10s}: {m["n_clusters"]} clusters, '
              f'{m["noise_share"] * 100:.1f}% noise')

print(f'\nPoint-level scored: {n_points_written:,}')
print(f'Stability (mean Jaccard top-1% clean window-level): {mean_jaccard_clean_1pct:.2f}')

print(f'\nArtifacts:')
joblib_files = sorted(glob.glob(os.path.join(ARTIFACTS_DIR, 'hdb_*.joblib')))
for f in joblib_files:
    size_kb = os.path.getsize(f) / 1e3
    print(f'  OK   {os.path.basename(f):<40s} ({size_kb:>6.0f} KB)')
print(f'  OK   {os.path.basename(HDB_WINDOW_SCORES_PATH):<40s} '
      f'({os.path.getsize(HDB_WINDOW_SCORES_PATH) / 1e6:>6.1f} MB)')
print(f'  OK   {os.path.basename(HDB_POINT_SCORES_PATH):<40s} '
      f'({os.path.getsize(HDB_POINT_SCORES_PATH) / 1e6:>6.1f} MB)')
print(f'  OK   {os.path.basename(HDB_STABILITY_PATH):<40s}')

print('\n03.3 complete. Ready for 03.4 (LSTM-AE).')


03.3 SUMMARY

Windows total: 2,505,273
Per phase group:
       climb: 466,518 windows (438,228 finite)
      cruise: 1,264,033 windows (1,252,100 finite)
     descent: 441,380 windows (437,127 finite)
    terminal: 318,809 windows (305,892 finite)

HDBSCAN models trained:
       climb: 2 clusters, 26.7% noise
      cruise: 3 clusters, 21.3% noise
     descent: 2 clusters, 66.6% noise
    terminal: 96 clusters, 90.0% noise

Point-level scored: 63,019,803
Stability (mean Jaccard top-1% clean window-level): 0.7726

Artifacts:
  ✓ hdb_climb_main.joblib                    ( 30440 KB)
  ✓ hdb_climb_s2.joblib                      ( 30445 KB)
  ✓ hdb_climb_s3.joblib                      ( 30442 KB)
  ✓ hdb_cruise_main.joblib                   ( 30491 KB)
  ✓ hdb_cruise_s2.joblib                     ( 30494 KB)
  ✓ hdb_cruise_s3.joblib                     ( 30482 KB)
  ✓ hdb_descent_main.joblib                  ( 30440 KB)
  ✓ hdb_descent_s2.joblib                    ( 30451 KB)
  ✓ hdb_descen